# Baseline simples — Churn Telco (IBM)

Notebook de apoio ao **Tech Challenge**: mesma ideia do [`Baseline`](../src/services/pipelines/baseline.py) da API, mas **mínima** e legível na banca.

- **DummyClassifier** (`strategy="stratified"`): referência trivial — prevê proporcionalmente às classes (melhor que "sempre a mesma classe" para métricas de calibragem).
- **LogisticRegression** (`class_weight="balanced"`): baseline linear forte, comparável ao pipeline de produção.

**Contrato alinhado ao projeto**: alvo binário `target` na **última coluna** após preparação; sem coluna de ID no treino.

In [ ]:
from __future__ import annotations

import urllib.request
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore", category=UserWarning)


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "pyproject.toml").is_file() and (p / "src").is_dir():
            return p
    return here


REPO = find_repo_root()
print("REPO =", REPO)

## 1. Carregar dados

Ordem: ficheiros locais (como no README) → descarga do mirror IBM no GitHub se faltar.

In [ ]:
LOCAL_CSV_NAMES = (
    "WA_Fn-UseC_-Telco-Customer-Churn.csv",
    "Telco-Customer-Churn.csv",
)
REMOTE_TELCO_CSV = (
    "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
    "master/data/Telco-Customer-Churn.csv"
)

search_dirs = [
    REPO / "ml_data" / "uploads",
    REPO / "src" / "data",
    REPO / "uploads",
]

csv_path: Path | None = None
for d in search_dirs:
    for name in LOCAL_CSV_NAMES:
        cand = d / name
        if cand.is_file():
            csv_path = cand
            break
    if csv_path:
        break

if csv_path is None:
    dest = REPO / "ml_data" / "uploads"
    dest.mkdir(parents=True, exist_ok=True)
    csv_path = dest / "Telco-Customer-Churn.csv"
    print("A descarregar CSV de exemplo a partir do mirror IBM...")
    urllib.request.urlretrieve(REMOTE_TELCO_CSV, csv_path)

print("CSV:", csv_path)
df = pd.read_csv(csv_path)
df.shape

## 2. Preparação (simples)

- Remove `customerID` se existir.
- `TotalCharges` → numérico (espaços vazios → NaN, depois imputer na pipeline).
- `Churn` → `target` 0/1 na **última coluna**.

In [ ]:
id_cols = [c for c in df.columns if c.lower() == "customerid"]
df = df.drop(columns=id_cols, errors="ignore")

if "Churn" not in df.columns:
    raise ValueError("Coluna 'Churn' esperada no dataset Telco.")

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

y = df["Churn"].map({"Yes": 1, "No": 0, 1: 1, 0: 0}).astype(int)
X = df.drop(columns=["Churn"])

X["target"] = y.values
X = X[[c for c in X.columns if c != "target"] + ["target"]]

y = X.pop("target")
print(X.shape, y.value_counts(normalize=True).round(3))

## 3. Pré-processamento + modelos

Mesma transformação para ambos; só muda o classificador final.

In [ ]:
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[("imputer", SimpleImputer(strategy="median"))]
            ),
            numeric_features,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    (
                        "onehot",
                        OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                    ),
                ]
            ),
            categorical_features,
        ),
    ]
)

RANDOM_STATE = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

models = {
    "Dummy (stratified)": DummyClassifier(strategy="stratified", random_state=RANDOM_STATE),
    "LogisticRegression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
}

rows = []
for name, clf in models.items():
    pipe = Pipeline([("prep", preprocess), ("clf", clf)])
    pipe.fit(X_train, y_train)
    proba = (
        pipe.predict_proba(X_test)[:, 1]
        if hasattr(pipe.named_steps["clf"], "predict_proba")
        else None
    )
    pred = pipe.predict(X_test)
    row = {
        "modelo": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
    }
    if proba is not None:
        row["roc_auc"] = roc_auc_score(y_test, proba)
    else:
        row["roc_auc"] = np.nan
    rows.append(row)

summary = pd.DataFrame(rows).set_index("modelo").round(4)
summary

## 4. Leitura rápida

- O **dummy estratificado** costuma ter ROC-AUC ~0.5 e F1 baixo: serve para provar que o regressão logística **melhora** face ao “chute” proporcional às classes.
- O pipeline completo da API acrescenta MLflow, EDA gravada, `manifest.json` e alinhamento com o `/processor` — ver `Baseline` em `src/services/pipelines/baseline.py`.